# Rakuten Product Image Classification

The goal of this notebook is to first preprocess the images and then apply a model to it (here efficientnet_b0) to predict product classes from product images.

# Setup and Imports

In [1]:
import os
os.chdir(os.path.expanduser('~/Desktop/Alles/Liora/Rakuten_Project')) # change to your specific path
BASE_DIR = os.getcwd()
IMAGES_DIR = os.path.join(BASE_DIR, 'images')
DATA_DIR = os.path.join(BASE_DIR, 'text')
import torch
device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
print(device)

import timm
import pandas as pd
import numpy as np
from PIL import Image
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset
from torchvision import transforms
from torch.utils.data import DataLoader
from sklearn.metrics import f1_score

mps


# Load Data

Here is an example of a raw image before cropping

In [2]:
img = Image.open(IMAGES_DIR + '/image_train/image_482853_product_928819.jpg')
img

Some images have a big white space around the product. We need to crop and resize the images for further processing.


# Image Preprocessing


## Defining a Cropping function

The cropping works as follows: the average pixel value of the four corners is calculated as a reference for the background color. 
A threshold of 85% of this corner average is used — pixels darker than this threshold are considered part of the product. The image is then cropped to the bounding box of these pixels. If no dark pixels are found, or if the bounding box has no area, the image is simply resized to 224x224 without cropping.
This allows dynamic cropping regardless of background color. The images are resized to 224x224 to be compatible with pretrained models like EfficientNet-B0.

In [3]:
def cropping(image):
    
    
    image = image.convert('RGB')
    img_array = np.array(image)
    top_left = img_array[0,0].mean()
    top_right = img_array[0,-1].mean()
    bottom_left = img_array[-1,0].mean()
    bottom_right = img_array[-1,-1].mean()
    corners = (top_left + top_right + bottom_left + bottom_right) / 4
    threshold = corners * 0.85
    rows, cols, _ = np.where(img_array < threshold)
    if rows.size == 0:
       res = image.resize((224, 224))
    else:
        top, bottom = rows.min(), rows.max()
        left, right = cols.min(), cols.max()
        if top == bottom or left == right:
            res = image.resize((224, 224))
        else:
            res = image.crop((left, top, right, bottom))
            res = res.resize((224, 224))
    return res

In [4]:
cropping(img)

## Cropping all images

I created a new folder which will hold all the cropped images. The loop then applies the cropping function to each image and saves it in said folder.

In [ ]:
os.makedirs('images/cropped_image_train')
file_names = os.listdir(IMAGES_DIR + '/image_train')
for i in file_names:
    img = Image.open(os.path.join('images/image_train/',i))
    cropped_img = cropping(img)
    cropped_img.save(os.path.join('images/cropped_image_train/',i))

# Image Processing

I loaded the preprocessed data frame from the text data and added the image path to the corresponding image. Since I only kept the products which text was identified as french or english. The data frame does not have all the 84k rows. After filtering it consists of 77344 rows and 7 columns (designation, description, productid, imageid, prdtypecode, text and language)

In [5]:
IMAGES_CROPPED_DIR = os.path.join(BASE_DIR, 'images', 'cropped_image_train')
preprocessed_df_image = pd.read_csv(DATA_DIR + '/df_preprocessed.csv')
preprocessed_df_image['image_path'] = IMAGES_CROPPED_DIR +'/image_' + preprocessed_df_image['imageid'].astype(str) + '_product_' + preprocessed_df_image['productid'].astype(str) +'.jpg'


In [6]:
# A check to see if every row has an existing path
preprocessed_df_image['file_exists'] = preprocessed_df_image['image_path'].apply(lambda x: os.path.exists(x))
print(preprocessed_df_image['file_exists'].value_counts())
preprocessed_df_image.drop(columns=['file_exists'], inplace=True)


file_exists
True    77344
Name: count, dtype: int64


## Train Test Split

Here you can either use the whole data frame with all its 77344 rows or you can use a subsample of the data to check if the pipeline works. The class labels are not in the range 0 to 26 which PyTorch requires. LabelEncoder maps the original class codes (e.g. 10, 40, 2280) to consecutive integers starting from 0 (0, 1, 2, ... 26). In train_test_split stratify = y_image_encoded is needed since we don't have an equal distribution of the classes.

In [7]:

#test_sample = preprocessed_df_image.sample(20000)


X_image = preprocessed_df_image['image_path'] # with preprocessed_df_image all rows will be analyized!! You can change it to test_sample 
y_image = preprocessed_df_image['prdtypecode'] # with preprocessed_df_image all rows will be analyized!! You can change it to test_sample 

le = LabelEncoder()
y_image_encoded = le.fit_transform(y_image)

X_train_img, X_val_img, y_train_img, y_val_img = train_test_split(X_image, y_image_encoded, test_size=0.25, random_state=42, stratify=y_image_encoded)

In [8]:
# Double check to see if the encoding worked
print("Min Label:", y_train_img.min(), "Max Label:", y_train_img.max())
print("Number of unique Labels:", len(set(y_train_img)))

Min Label: 0 Max Label: 26
Number of unique Labels: 27


## Model Preparation

### Dataset, Transformer and DataLoader

PyTorch requires a custom Dataset class to load the images. 
RakutenDataset inherits from PyTorch's Dataset class and implements 
three methods:

- `__init__`: stores the image paths, labels and optional transforms
- `__len__`: returns the total number of images
- `__getitem__`: opens a single image by index, converts it to RGB, 
  applies transforms and returns the image-label pair

The DataLoader then uses this class to automatically load images 
in batches during training.

The transform pipeline prepares each image for the model
- `Resize` resizes to 224 by 224 pixel
- `ToTensor` changes the image from a PIL format to a PyTorch tensor and scales the pixel values from [0, 255] to [0.0, 1.0]
- `Normalize` normalizes the pixel values based on the mean and std of the pre trained model of each channel (RGB)

In [9]:
from torch.utils.data import Dataset

class RakutenDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths.tolist()
        self.labels = labels.tolist()
        self.transform = transform
    
    def __len__(self):
        return len(self.image_paths) 
    
    def __getitem__(self, idx):
        image = Image.open(self.image_paths[idx]).convert('RGB')
        label = self.labels[idx]
        
        if self.transform:
            image = self.transform(image)
        
        return image, label


from torchvision import transforms

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                         std=[0.229, 0.224, 0.225])
])

I now instantiate train_dataset and val_dataset as instances of RakutenDataset

In [10]:
train_dataset = RakutenDataset(
    image_paths=X_train_img,
    labels=y_train_img,
    transform=transform
)


val_dataset = RakutenDataset(
    image_paths=X_val_img,
    labels=y_val_img,
    transform=transform
)

Now I used DataLoader to set the `batch_size` and `shuffle` the data for the train_loader. The val_loader is not shuffled since the order does not matter. The model does not learn from the validation batches but only evaluates its predictions.
DataLoader then gets batches of 32 images and also the corresponding labels.

In [11]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False
)

## Model

I use EfficientNet-B0 as the pretrained model instead of the benchmark's ResNet50. EfficientNet-B0 is known for being computationally efficient while maintaining strong performance.
The model is loaded with pretrained ImageNet weights (`pretrained=True`) and the final layer is set to 27 outputs, one per product category.
The optimizer's learning rate is set to `lr=0.0001`. Compared to `lr=0.001`, this smaller learning rate reduces overfitting and allows the model to learn more stably over multiple epochs.

In [12]:
model_b0 = timm.create_model('efficientnet_b0', pretrained=True, num_classes=27)
model_b0 = model_b0.to(device)

criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model_b0.parameters(), lr = 0.0001)

The training loop runs for a maximum of 20 epochs. Each epoch consists 
of two phases:

Training phase: The model processes all training batches. For each batch it makes predictions, calculates the loss, and updates the weights via backpropagation.  
Validation phase: The model evaluates on the validation set without updating any weights. The weighted F1 score is calculated at the end of each epoch.  
Early Stopping: If the validation loss does not improve for 3 consecutive epochs, training is stopped to prevent overfitting.  
Model saving: The model with the best F1 score is saved to `best_model.pth` — not necessarily the last epoch, but the one that performed best on the validation set.

Note: these results were obtained training on the full 77344 rows.
Epoch 4 had the highest f1 score and was therefore saved.

Epoch 4/20 | 
**Train Loss:** 0.5370 |
**Val Loss:** 1.3107 |
**Val Accuracy:** 0.6415 |
**F1 Score:** 0.6382

In [ ]:
num_epochs = 20
best_val_loss = float('inf')  # "unendlich" als Startwert
patience = 3                   # wie viele Epochen warten wir noch?
patience_counter = 0  
best_f1 = 0
for epoch in range(num_epochs):
    # Training
    model_b0.train()
    train_loss = 0
    
    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)
        
        optimizer.zero_grad()
        outputs = model_b0(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    
    # Validation
    model_b0.eval()
    val_loss = 0
    correct = 0
    total = 0
    all_labels = []
    all_predicted = []   
    
    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)
            
            outputs = model_b0(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            all_labels.extend(labels.cpu().numpy())
            all_predicted.extend(predicted.cpu().numpy())
    f1 = f1_score(all_labels, all_predicted, average='weighted')
    print(f"Epoch {epoch+1}/{num_epochs} | Train Loss: {train_loss/len(train_loader):.4f} | Val Loss: {val_loss/len(val_loader):.4f} | Val Accuracy: {correct/total:.4f} | F1 Score: {f1}" )
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print("Early Stopping!")
            break

    if f1 > best_f1:
        best_f1 = f1
        torch.save(model_b0.state_dict(), 'best_model.pth')